# Spam Email Detection

## 05. Hyperparameter Tuning

GridSearchCV and Stratified K-Fold Cross Validation are used to identify suitable hyperparameters for the Logistic Regression text-classification pipeline.

In [2]:
## output

from pathlib import Path

# Project root
BASE_DIR = Path.cwd().parent.parent

# Dataset
DATA_DIR = BASE_DIR / "data"

# Outputs
OUTPUT_DIR = BASE_DIR / "outputs"
FIGURE_DIR = OUTPUT_DIR / "figures"
RESULT_DIR = OUTPUT_DIR / "results"

# Models
MODEL_DIR = BASE_DIR / "models"

# Create folders
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = DATA_DIR / "email.csv"

print("PROJECT:", BASE_DIR)
print("DATASET:", DATA_PATH)
print("FIGURES:", FIGURE_DIR)
print("RESULTS:", RESULT_DIR)
print("MODELS:", MODEL_DIR)

PROJECT: c:\Users\HP\OneDrive\Desktop\Spam-Email-Detection
DATASET: c:\Users\HP\OneDrive\Desktop\Spam-Email-Detection\data\email.csv
FIGURES: c:\Users\HP\OneDrive\Desktop\Spam-Email-Detection\outputs\figures
RESULTS: c:\Users\HP\OneDrive\Desktop\Spam-Email-Detection\outputs\results
MODELS: c:\Users\HP\OneDrive\Desktop\Spam-Email-Detection\models


In [3]:
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    StratifiedKFold
)

from sklearn.pipeline import Pipeline

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.feature_selection import SelectKBest, chi2

from sklearn.linear_model import LogisticRegression

BASE_DIR = Path.cwd().parent.parent

DATA_PATH = BASE_DIR / "data" / "email.csv"

RESULT_DIR = BASE_DIR / "outputs" / "results"

RESULT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)

df["Category"] = df["Category"].astype(str).str.strip().str.lower()

df = df[
    df["Category"].isin(["ham", "spam"])
].copy()

df["Message"] = df["Message"].astype(str)

df = df.drop_duplicates()

df["label"] = df["Category"].map({
    "ham": 0,
    "spam": 1
})

X = df["Message"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [4]:
pipeline = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            stop_words="english"
        )
    ),

    (
        "select",
        SelectKBest(
            chi2
        )
    ),

    (
        "model",
        LogisticRegression(
            max_iter=2500,
            random_state=42
        )
    )
])

In [5]:
param_grid = {

    "tfidf__ngram_range": [
        (1, 1),
        (1, 2)
    ],

    "tfidf__min_df": [
        1,
        2
    ],

    "select__k": [
        1000,
        2000,
        3000
    ],

    "model__C": [
        0.5,
        1.0,
        2.0
    ],

    "model__class_weight": [
        None,
        "balanced"
    ]
}

In [6]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(
    X_train,
    y_train
)

print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest Cross Validation F1:")
print(grid_search.best_score_)

Fitting 5 folds for each of 72 candidates, totalling 360 fits
Best Parameters:
{'model__C': 2.0, 'model__class_weight': 'balanced', 'select__k': 2000, 'tfidf__min_df': 2, 'tfidf__ngram_range': (1, 1)}

Best Cross Validation F1:
0.9215605923493315


In [7]:
cv_results = pd.DataFrame(
    grid_search.cv_results_
)

cv_results = cv_results.sort_values(
    "rank_test_score"
)

cv_results.to_csv(
    RESULT_DIR / "15_hyperparameter_tuning_results.csv",
    index=False
)

best_parameters = pd.DataFrame(
    [grid_search.best_params_]
)

best_parameters.to_csv(
    RESULT_DIR / "16_best_hyperparameters.csv",
    index=False
)

best_parameters

,model__C,model__class_weight,select__k,tfidf__min_df,tfidf__ngram_range
0,2.0,balanced,2000,2,"(1, 1)"
